#### Making the prompts for LLM training

In [30]:
%pip install shap

Note: you may need to restart the kernel to use updated packages.


In [7]:
import pandas as pd
import shap
import numpy as np
from xgboost import XGBClassifier


In [8]:
data=pd.read_csv("/teamspace/studios/this_studio/NLP-Projects/LLM-for-IDS-log-analysis/Data/Classifier_Data/bot_iot_dataset_preprocessed.csv")
data.head()

,stime,flgs_e,flgs_e s,flgs_e dS,flgs_e g,flgs_e *,flgs_eU,flgs_e &,flgs_e d,flgs_e F,...,tnp_per_dport,ar_p_proto_p_srcip,ar_p_proto_p_dstip,n_in_conn_p_srcip,n_in_conn_p_dstip,ar_p_proto_p_sport,ar_p_proto_p_dport,pkts_p_state_p_protocol_p_destip,pkts_p_state_p_protocol_p_srcip,attack_type
0,1.526963e+09,1,0,0,0,0,0,0,0,0,...,8,4918.426175,939.965691,641.0,5208,564.567137,10430.133552,6482,23345,3
1,1.528081e+09,0,1,0,0,0,0,0,0,0,...,13108094,3908.819243,1254.250649,120.0,6669,0.094425,100.862697,12994,15429,1
2,1.526982e+09,1,0,0,0,0,0,0,0,0,...,21,3908.819243,1254.250649,120.0,6669,4368.510051,18097.726590,13229,23301,3
3,1.528081e+09,0,1,0,0,0,0,0,0,0,...,13108094,4574.700595,9901.705447,146.0,5063,0.191534,100.862697,14111,30428,1
4,1.526982e+09,1,0,0,0,0,0,0,0,0,...,23,4918.426175,1254.250649,641.0,6669,1856.201331,5937.703676,13229,23345,3


In [18]:
features=['flgs_e', 'flgs_e s', 'flgs_e dS', 'flgs_e g', 'flgs_e *', 'flgs_eU',
       'flgs_e &', 'flgs_e    F', 'flgs_e r', 'proto_tcp', 'proto_udp',
       'proto_icmp', 'proto_arp', 'proto_ipv6-icmp', 'proto_rarp',
       'proto_igmp', 'saddr_192.168.100.149', 'saddr_192.168.100.148',
       'saddr_192.168.100.150', 'saddr_192.168.100.147', 'saddr_192.168.100.3',
       'saddr_192.168.100.7', 'saddr_192.168.100.5', 'saddr_192.168.100.4',
       'saddr_192.168.100.1', 'daddr_192.168.100.149', 'daddr_192.168.100.150',
       'daddr_192.168.100.147', 'daddr_192.168.100.3', 'daddr_192.168.100.7',
       'daddr_192.168.100.6', 'daddr_192.168.100.5', 'daddr_192.168.100.4',
       'daddr_private', 'daddr_external', 'pkts', 'bytes', 'state_RST',
       'state_CON', 'state_NRS', 'state_ACC', 'state_MAS', 'dur', 'mean',
       'min', 'dpkts', 'dbytes', 'tnp_per_dport', 'ar_p_proto_p_dstip',
       'ar_p_proto_p_sport', 'pkts_p_state_p_protocol_p_destip','attack_type']

In [17]:
len(data)*0.061

3008.398

In [20]:
data_sample=data.sample(frac=0.061).reset_index()
data_sample=data_sample[features].iloc[:3000]
data_sample.head()

,flgs_e,flgs_e s,flgs_e dS,flgs_e g,flgs_e *,flgs_eU,flgs_e &,flgs_e F,flgs_e r,proto_tcp,...,dur,mean,min,dpkts,dbytes,tnp_per_dport,ar_p_proto_p_dstip,ar_p_proto_p_sport,pkts_p_state_p_protocol_p_destip,attack_type
0,1,0,0,0,0,0,0,0,0,1,...,0.036125,0.036125,0.036125,1,60,2,5274.167447,33.222485,65132,4
1,1,0,0,0,0,0,0,0,0,1,...,8.813676,0.001685,0.000000,1,60,13108094,5274.167447,0.340380,65132,2
2,1,0,0,0,0,0,0,0,0,1,...,33.937874,0.000020,0.000000,4,766,13108094,5274.167447,0.265190,65132,1
3,0,0,0,1,0,0,0,0,0,1,...,23.880112,0.093670,0.043778,2,120,13108094,5274.167447,0.169986,65132,1
4,1,0,0,0,0,0,0,0,0,0,...,5.047973,0.000000,0.000000,0,0,14770,0.201614,0.198099,5006,0


In [75]:
metadata={'pkseqid': 'Row Identifier',
 'stime': 'Record start time',
 'flgs_e': 'State ESTABLISHED',
 'flgs_e s' : 'State ESTABLISHED and State SYN_SENT',
 'flgs_e dS' : 'State ESTABLISHED and State SYN_RECEIVED',
 'flgs_e g' : 'State ESTABLISHED and State FIN_WAIT_1',
 'flgs_e *' : 'State ESTABLISHED and State ACK',
 'flgs_eU' : 'State ESTABLISHED and State URGENT',
 'flgs_e &' : 'State ESTABLISHED and State PUSH',
 'flgs_e    F' : 'State ESTABLISHED and State FIN_WAIT_2',
 'flgs_e r' : 'State ESTABLISHED and State RESET',
 'proto_tcp': 'TCP',
 'proto_udp': 'UDP',
 'proto_icmp': 'ICMP',
 'proto_arp': 'ARP',
 'proto_ipv6-icmp': 'IPV6-ICMP',
 'proto_rarp': 'RARP',
 'proto_igmp': 'IGMP',
 'saddr_192.168.100.149': '192.168.100.149',
 'saddr_192.168.100.148': '192.168.100.148',
 'saddr_192.168.100.150': '192.168.100.150',
 'saddr_192.168.100.147': '192.168.100.147',
 'saddr_192.168.100.1': '192.168.100.1',
 'saddr_192.168.100.3': '192.168.100.3',
 'saddr_192.168.100.4': '192.168.100.4',
 'saddr_192.168.100.7': '192.168.100.7',
 'saddr_192.168.100.5': '192.168.100.5',
 'daddr_192.168.100.149': '192.168.100.149',
 'daddr_192.168.100.148': '192.168.100.148',
 'daddr_192.168.100.150': '192.168.100.150',
 'daddr_192.168.100.147': '192.168.100.147',
 'daddr_192.168.100.1': '192.168.100.1',
 'daddr_192.168.100.3': '192.168.100.3',
 'daddr_192.168.100.4': '192.168.100.4',
 'daddr_192.168.100.7': '192.168.100.7',
 'daddr_192.168.100.6': '192.168.100.6',
 'daddr_192.168.100.5': '192.168.100.5',
 'daddr_private' : 'Private',
 'daddr_external' : 'External',
 'pkts': 'Total count of packets in transaction',
 'bytes': 'Total number of bytes in transaction',
 'state': 'Transaction state',
 'state_RST' : 'RESET',
 'state_CON' : 'CONNECTED',
 'state_NRS' : 'NO_RESPONSE',
 'state_ACC' : 'ACCEPTED',
 'state_MAS' : 'MASQUERADE',
 'ltime': 'Record last time',
 'seq': 'Argus sequence number',
 'dur': 'Record total duration',
 'mean': 'Average duration of aggregated records',
 'stddev': 'Standard deviation of aggregated records',
 'sum': 'Total duration of aggregated records',
 'min': 'Minimum duration of aggregated records',
 'max': 'Maximum duration of aggregated records',
 'spkts': 'Source-to-destination packet count',
 'dpkts': 'Destination-to-source packet count',
 'sbytes': 'Source-to-destination byte count',
 'dbytes': 'Destination-to-source byte count',
 'rate': 'Total packets per second in transaction',
 'srate': 'Source-to-destination packets per second',
 'drate': 'Destination-to-source packets per second',
 'tnbpsrcip': 'Total Number of bytes per source IP',
 'tnbpdstip': 'Total Number of bytes per Destination IP',
 'tnp_psrcip': 'Total Number of packets per source IP',
 'tnp_pdstip': 'Total Number of packets per Destination IP',
 'tnp_perproto': 'Total Number of packets per protocol',
 'tnp_per_dport': 'Total Number of packets per destination port',
 'ar_p_proto_p_srcip': 'Average rate per protocol per Source IP',
 'ar_p_proto_p_dstip': 'Average rate per protocol per Destination IP',
 'n_in_conn_p_srcip': 'Number of inbound connections per source IP',
 'n_in_conn_p_dstip': 'Number of inbound connections per destination IP',
 'ar_p_proto_p_sport': 'Average rate per protocol per source port',
 'ar_p_proto_p_dport': 'Average rate per protocol per destination port',
 'pkts_p_state_p_protocol_p_destip': 'Number of packets grouped by state of flows and protocols per destination IP',
 'pkts_p_state_p_protocol_p_srcip': 'Number of packets grouped by state of flows and protocols per source IP',
 'attack': 'Class label: 0 for Normal traffic, 1 for Attack Traffic',
 'category': 'Traffic category',
 'subcategory ': 'Traffic subcategory', 
 'attack_type' : 'Numeric representation of Traffic category and subcategory (0 for Normal, 1 for DoS, 2 for DDoS, 3 for OS_Fingerprint, 4 for Service_Scan, 5 for Keylogging, 6 for Data_Exfiltration)'
  }

In [32]:
features=['flgs_e', 'flgs_e s', 'flgs_e dS', 'flgs_e g', 'flgs_e *', 'flgs_eU',
       'flgs_e &', 'flgs_e    F', 'flgs_e r', 'proto_tcp', 'proto_udp',
       'proto_icmp', 'proto_arp', 'proto_ipv6-icmp', 'proto_rarp',
       'proto_igmp', 'saddr_192.168.100.149', 'saddr_192.168.100.148',
       'saddr_192.168.100.150', 'saddr_192.168.100.147', 'saddr_192.168.100.3',
       'saddr_192.168.100.7', 'saddr_192.168.100.5', 'saddr_192.168.100.4',
       'saddr_192.168.100.1', 'daddr_192.168.100.149', 'daddr_192.168.100.150',
       'daddr_192.168.100.147', 'daddr_192.168.100.3', 'daddr_192.168.100.7',
       'daddr_192.168.100.6', 'daddr_192.168.100.5', 'daddr_192.168.100.4',
       'daddr_private', 'daddr_external', 'pkts', 'bytes', 'state_RST',
       'state_CON', 'state_NRS', 'state_ACC', 'state_MAS', 'dur', 'mean',
       'min', 'dpkts', 'dbytes', 'tnp_per_dport', 'ar_p_proto_p_dstip',
       'ar_p_proto_p_sport', 'pkts_p_state_p_protocol_p_destip']

In [33]:
X_train=data.sample(frac=0.6)
X_test=data.sample(frac=0.4)
y_train=X_train["attack_type"]
y_test=X_test["attack_type"]
X_train.drop("attack_type",axis=1,inplace=True)
X_test.drop("attack_type",axis=1,inplace=True)
X_train=X_train[features]
X_test=X_test[features]

In [34]:
model=XGBClassifier()
model.fit(X_train,y_train)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric=None, feature_types=None,
              gamma=None, grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=None, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=None, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=None, n_jobs=None,
              num_parallel_tree=None, objective='multi:softprob', ...)

In [35]:
model.score(X_test,y_test)

0.9883915445835657

In [90]:
explainer=shap.TreeExplainer(model)
shap_values=explainer.shap_values(data_sample.drop("attack_type",axis=1))

In [ ]:
#prompt example : Network Traffic (features), Classifier Decision: Attack, SHAP Values: Feature A (+0.3), Feature B (-0.2), Feature C (+0.1)

In [71]:
def get_key(dict,value) :
    return [key for key in dict.keys() if dict[key]==value][0]
def sort_get_best(shap_values,features) :
    feat_dict={features[i]:shap_values[i] for i in range(0,len(features))}
    sorted_shap = np.argsort(np.abs(shap_values))
    result_shap = shap_values[sorted_shap][::-1][:10]
    feature_importances={get_key(feat_dict,value):value for value in result_shap}
    return feature_importances
    

In [89]:
y_preds=model.predict(data_sample.drop("attack_type",axis=1))

In [56]:
classes={0:'Normal Traffic',1 : 'DoS attack',2: 'DDoS attack',3: 'OS Fingerprint attack',4: 'Service Scan attack',5 : 'Keylogging attack',6 : 'Data Exfiltration attack'}

In [62]:
data_sample.columns

Index(['flgs_e', 'flgs_e s', 'flgs_e dS', 'flgs_e g', 'flgs_e *', 'flgs_eU',
       'flgs_e &', 'flgs_e    F', 'flgs_e r', 'proto_tcp', 'proto_udp',
       'proto_icmp', 'proto_arp', 'proto_ipv6-icmp', 'proto_rarp',
       'proto_igmp', 'saddr_192.168.100.149', 'saddr_192.168.100.148',
       'saddr_192.168.100.150', 'saddr_192.168.100.147', 'saddr_192.168.100.3',
       'saddr_192.168.100.7', 'saddr_192.168.100.5', 'saddr_192.168.100.4',
       'saddr_192.168.100.1', 'daddr_192.168.100.149', 'daddr_192.168.100.150',
       'daddr_192.168.100.147', 'daddr_192.168.100.3', 'daddr_192.168.100.7',
       'daddr_192.168.100.6', 'daddr_192.168.100.5', 'daddr_192.168.100.4',
       'daddr_private', 'daddr_external', 'pkts', 'bytes', 'state_RST',
       'state_CON', 'state_NRS', 'state_ACC', 'state_MAS', 'dur', 'mean',
       'min', 'dpkts', 'dbytes', 'tnp_per_dport', 'ar_p_proto_p_dstip',
       'ar_p_proto_p_sport', 'pkts_p_state_p_protocol_p_destip',
       'attack_type'],
      dtype='obje

In [84]:
data_6=data[data["attack_type"]==6][data_sample.columns]
data_5=data[data["attack_type"]==5][data_sample.columns].iloc[:600]
data_sample=pd.concat([data_sample,data_6,data_5],axis=0)

attack_type
5    1548
3     608
0     605
2     602
4     555
1     544
6     125
Name: count, dtype: int64

In [91]:
#making the prompts
flags_state=['flgs_e', 'flgs_e s', 'flgs_e dS', 'flgs_e g', 'flgs_e *', 'flgs_eU',
       'flgs_e &', 'flgs_e    F', 'flgs_e r']
protos=['proto_tcp', 'proto_udp',
       'proto_icmp', 'proto_arp', 'proto_ipv6-icmp', 'proto_rarp',
       'proto_igmp']
s_addr=['saddr_192.168.100.149', 'saddr_192.168.100.148',
       'saddr_192.168.100.150', 'saddr_192.168.100.147', 'saddr_192.168.100.3',
       'saddr_192.168.100.7', 'saddr_192.168.100.5', 'saddr_192.168.100.4',
       'saddr_192.168.100.1']
d_addr=['daddr_192.168.100.149', 'daddr_192.168.100.150',
       'daddr_192.168.100.147', 'daddr_192.168.100.3', 'daddr_192.168.100.7',
       'daddr_192.168.100.6', 'daddr_192.168.100.5', 'daddr_192.168.100.4']
d_type=['daddr_private','daddr_external']
states=['state_RST','state_CON', 'state_NRS', 'state_ACC', 'state_MAS']
transaction_info=['pkts', 'bytes','dur', 'mean',
       'min', 'dpkts', 'dbytes', 'tnp_per_dport', 'ar_p_proto_p_dstip',
       'ar_p_proto_p_sport', 'pkts_p_state_p_protocol_p_destip']
dict_llm={}
for i in range(len(data_sample)) :
    dict_llm[i]={}
    #network traffic and classifier decision
    prompt="Network Traffic :\n\n"
    data_row=data_sample.iloc[i]
    for feature in data_row.index :
        if feature=="attack_type" :
           prompt=prompt+f"\nClassifier Decision : {classes[y_preds[i]]}\n"
        elif feature in flags_state :
            if data_row[feature]==1.0 :
                prompt=prompt+f"Flow state flags: {metadata[feature]}\n"
        elif feature in protos :
            if data_row[feature]==1.0 :
                prompt=prompt+f"Transaction protocol : {metadata[feature]}\n"
        elif feature in s_addr :
            if data_row[feature]==1.0 :
                prompt=prompt+f"Source IP address : {metadata[feature]}\n"
        elif feature in d_addr :
            if data_row[feature]==1.0 :
                prompt=prompt+f"Destination IP address : {metadata[feature]}\n"
        elif feature in d_type :
            if data_row[feature]==1.0 :
                prompt=prompt+f"Destination address type : {metadata[feature]}\n"
        elif feature in states :
            if data_row[feature]==1.0 :
                prompt=prompt+f"-Transaction State : {metadata[feature]}\n"
        elif feature in transaction_info :
            if "\nTransaction information :\n" in prompt :
                prompt=prompt+f"-{metadata[feature]} : {round(data_row[feature],2)}\n"
            else :
                prompt=prompt+"\nTransaction information :\n"
                prompt=prompt+f"-{metadata[feature]} : {round(data_row[feature],2)}\n"
    #shap values
    prompt=prompt+"\nSHAP Values that represent how much a particular feature influenced the final decision:\n"
    best_shap_values=sort_get_best(shap_values[i,:,y_preds[i]],features)
    for feature in best_shap_values.keys() :
        if feature in flags_state :
           prompt=prompt+ f"-SHAP Value for Flow state flags : {metadata[feature]} : {round(best_shap_values[feature],2)} (Feature value: {data_row[feature]})\n"
        elif feature in protos :
           prompt=prompt+ f"-SHAP Value for Transaction protocol : {metadata[feature]} : {round(best_shap_values[feature],2)}. (Feature value: {data_row[feature]})\n"
        elif feature in s_addr :
           prompt=prompt+ f"-SHAP Value for Source IP Address ({metadata[feature]}) : {round(best_shap_values[feature],2)}. (Feature value: {data_row[feature]})\n"
        elif feature in d_addr :
           prompt=prompt+ f"-SHAP Value for Destination IP Address ({metadata[feature]}) : {round(best_shap_values[feature],2)}. (Feature value: {data_row[feature]})\n"
        elif feature in d_type :
           prompt=prompt+ f"-SHAP Value for Destination Address Type ({metadata[feature]}) : {round(best_shap_values[feature],2)}. (Feature value: {data_row[feature]})\n"
        elif feature in states :
           prompt=prompt+ f"-SHAP Value for Transaction state ({metadata[feature]}) : {round(best_shap_values[feature],2)}. (Feature value: {data_row[feature]})\n"
        else :
           prompt=prompt+ f"-SHAP Value for {metadata[feature]} : {round(best_shap_values[feature],2)}. (Feature value: {data_row[feature]})\n"
    dict_llm[i]["prompt"]=prompt[:-1]
    print(f"{i} done.")

0 done.
1 done.
2 done.
3 done.
4 done.
5 done.
6 done.
7 done.
8 done.
9 done.
10 done.
11 done.
12 done.
13 done.
14 done.
15 done.
16 done.
17 done.
18 done.
19 done.
20 done.
21 done.
22 done.
23 done.
24 done.
25 done.
26 done.
27 done.
28 done.
29 done.
30 done.
31 done.
32 done.
33 done.
34 done.
35 done.
36 done.
37 done.
38 done.
39 done.
40 done.
41 done.
42 done.
43 done.
44 done.
45 done.
46 done.
47 done.
48 done.
49 done.
50 done.
51 done.
52 done.
53 done.
54 done.
55 done.
56 done.
57 done.
58 done.
59 done.
60 done.
61 done.
62 done.
63 done.
64 done.
65 done.
66 done.
67 done.
68 done.
69 done.
70 done.
71 done.
72 done.
73 done.
74 done.
75 done.
76 done.
77 done.
78 done.
79 done.
80 done.
81 done.
82 done.
83 done.
84 done.
85 done.
86 done.
87 done.
88 done.
89 done.
90 done.
91 done.
92 done.
93 done.
94 done.
95 done.
96 done.
97 done.
98 done.
99 done.
100 done.
101 done.
102 done.
103 done.
104 done.
105 done.
106 done.
107 done.
108 done.
109 done.
110 done.


In [95]:
data_llm=pd.DataFrame(dict_llm).T
data_llm.to_csv("/teamspace/studios/this_studio/NLP-Projects/LLM-for-IDS-log-analysis/Data/LLM_Data/data_v1.csv",index=False)

### Making the responses

In [25]:
import pandas as pd
import numpy as np

In [26]:
data_llm=pd.read_csv("/teamspace/studios/this_studio/NLP-Projects/LLM-for-IDS-log-analysis/Data/LLM_Data/data_v1.csv")

In [10]:
first=3435
end=3440
count=first
prompt=f"generate why the network traffic classifier decided on classifying each network traffic as normal or a type of attack, based on the information in the inputs especially the SHAP values, keep each explanation in one paragraph and use no bullet points and no titles or headers and also return to the line once not twice after each explanation. always begin the response with '{first}, The model classified this traffic as' and keep incrementing {first} until you reach {end-1}."
prompt=prompt+" return to the line only one time after generating each end of sentence token.\n"
for i in range(first,end) : 
    prompt=prompt+f"\n(input {count} : {data_llm.prompt.iloc[i]})"
    count+=1
print(prompt)

generate why the network traffic classifier decided on classifying each network traffic as normal or a type of attack, based on the information in the inputs especially the SHAP values, keep each explanation in one paragraph and use no bullet points and no titles or headers and also return to the line once not twice after each explanation. always begin the response with '3435, The model classified this traffic as' and keep incrementing 3435 until you reach 3439. return to the line only one time after generating each end of sentence token.

(input 3435 : Network Traffic :

Flow state flags: State ESTABLISHED
Transaction protocol : TCP
Source IP address : 192.168.100.3
Destination IP address : 192.168.100.149
Destination address type : Private

Transaction information :
-Total count of packets in transaction : 2.0
-Total number of bytes in transaction : 134.0
-Transaction State : RESET
-Record total duration : 0.0
-Average duration of aggregated records : 0.0
-Minimum duration of aggrega

### Merging the prompts with the responses

In [27]:
prompts=pd.read_csv("/teamspace/studios/this_studio/NLP-Projects/LLM-for-IDS-log-analysis/Data/LLM_Data/prompts_final.csv")

In [28]:
len(prompts),len(data_llm)

(3538, 3538)

In [29]:
data_llm.head()

,prompt
0,Network Traffic :\n\nFlow state flags: State E...
1,Network Traffic :\n\nFlow state flags: State E...
2,Network Traffic :\n\nFlow state flags: State E...
3,Network Traffic :\n\nFlow state flags: State E...
4,Network Traffic :\n\nFlow state flags: State E...


In [31]:
prompts.columns=["id","response"]

In [33]:
data_llm=pd.concat([data_llm,prompts],axis=1)

In [35]:
data_llm=data_llm[["id","prompt","response"]]
data_llm.head()

,id,prompt,response
0,0,Network Traffic :\n\nFlow state flags: State E...,The model correctly classified this as a Servi...
1,1,Network Traffic :\n\nFlow state flags: State E...,The model correctly classified this as a DDoS ...
2,2,Network Traffic :\n\nFlow state flags: State E...,The model correctly classified this as a DoS a...
3,3,Network Traffic :\n\nFlow state flags: State E...,The model correctly classified this as a DoS a...
4,4,Network Traffic :\n\nFlow state flags: State E...,"The model classified this as Normal Traffic, r..."


In [36]:
data_llm.to_csv("/teamspace/studios/this_studio/NLP-Projects/LLM-for-IDS-log-analysis/Data/LLM_Data/full_llm_data.csv",index=False)

In [31]:
import pandas as pd
data_llm=pd.read_csv("/teamspace/studios/this_studio/NLP-Projects/LLM-for-IDS-log-analysis/Data/LLM_Data/full_llm_data.csv")
data_llm.head()

,id,prompt,response
0,0,Network Traffic :\n\nFlow state flags: State E...,The model correctly classified this as a Servi...
1,1,Network Traffic :\n\nFlow state flags: State E...,The model correctly classified this as a DDoS ...
2,2,Network Traffic :\n\nFlow state flags: State E...,The model correctly classified this as a DoS a...
3,3,Network Traffic :\n\nFlow state flags: State E...,The model correctly classified this as a DoS a...
4,4,Network Traffic :\n\nFlow state flags: State E...,"The model classified this as Normal Traffic, r..."


In [34]:
data_llm[data_llm.prompt.duplicated()]

,id,prompt,response
41,41,Network Traffic :\n\nFlow state flags: State E...,The classifier marked the traffic as normal ba...
58,58,Network Traffic :\n\nFlow state flags: State E...,The classifier identified the traffic as norma...
97,97,Network Traffic :\n\nFlow state flags: State E...,The classifier identified the network traffic ...
141,141,Network Traffic :\n\nFlow state flags: State E...,The classifier categorized the traffic as norm...
195,195,Network Traffic :\n\nFlow state flags: State E...,The classifier classified the network traffic ...
...,...,...,...
3530,3530,Network Traffic :\n\nFlow state flags: State E...,The model classified this traffic as a Keylogg...
3531,3531,Network Traffic :\n\nFlow state flags: State E...,The model classified this traffic as a Keylogg...
3533,3533,Network Traffic :\n\nFlow state flags: State E...,The model classified this traffic as a Keylogg...
3535,3535,Network Traffic :\n\nFlow state flags: State E...,The model classified this traffic as a keylogg...


In [ ]:
data_llm.